In [6]:
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from scipy.stats import norm
from scipy.optimize import brentq
import warnings

from dataset import OptionsDataModule
from model import OptionNetModule

warnings.filterwarnings("ignore")

In [10]:
# --- Constants ---
CHECKPOINT_PATH = './logs/my_experiment/version_82/checkpoints/epoch=49-step=69750.ckpt'
DATA_DIR = "./data/108105"
BATCH_SIZE = 128
RISK_FREE_RATE = 0.04  # Assumed r
SELECTED_DATE = "2025-01-02"  # Set to None to use latest available date



In [17]:
df = pd.read_csv("data/108105/2025_C_options_data.csv")

df["sofr"].bfill(inplace=True)
sum(df["sofr"].isna())

377382

In [8]:
def black_scholes_call_price(S, K, T, r, sigma):
    """Calculates Black-Scholes Call Option Price"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

In [13]:
print("Loading Data Module...")
# Initialize DataModule to get fitted scalers
data_module = OptionsDataModule(DATA_DIR, batch_size=BATCH_SIZE)
data_module.setup("fit")

print(f"Loading Model from {CHECKPOINT_PATH}...")
model = OptionNetModule.load_from_checkpoint(CHECKPOINT_PATH)
model.eval()

device = "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)

print("Loading raw data for selected date...")
raw_df = pd.concat(
    [pd.read_csv(path) for path in sorted(glob.glob(f"{DATA_DIR}/*.csv"))],
    ignore_index=True,
)
raw_df["date"] = pd.to_datetime(raw_df["date"])

if raw_df.empty:
    raise ValueError(f"No rows found under {DATA_DIR}.")

target_date = pd.to_datetime(SELECTED_DATE) if SELECTED_DATE else raw_df["date"].max()
daily = raw_df[raw_df["date"] == target_date].copy()
if daily.empty:
    available = raw_df["date"].dt.date.drop_duplicates().sort_values().astype(str).tail(10).tolist()
    raise ValueError(
        f"No rows found for SELECTED_DATE={target_date.date()}. "
        f"Last available dates: {available}"
    )

S_fixed = float(daily["S"].median())
vix_fixed = float(daily["vix"].median())
hv_cols = [col for col in ["hv_10", "hv_14", "hv_30", "hv_60", "hv_91"] if col in daily.columns]
if not hv_cols:
    raise ValueError("No historical volatility columns found (expected hv_10, hv_14, hv_30, hv_60, hv_91).")

h_vol_values = daily[hv_cols].median().astype(float).to_dict()
if pd.isna(vix_fixed):
    raise ValueError(f"VIX is missing for date {target_date.date()}.")
if any(pd.isna(v) for v in h_vol_values.values()):
    raise ValueError(f"Historical vol is missing for date {target_date.date()} in columns {hv_cols}.")

print(f"Using date={target_date.date()}, S={S_fixed:.2f}, VIX={vix_fixed:.4f}")
print("Historical vol snapshot:", h_vol_values)

# --- Generate Grid ---
print("Generating Grid...")
k_min = 0.6 * S_fixed
k_max = 1.5 * S_fixed
moneyness_min = S_fixed / k_max
moneyness_max = S_fixed / k_min
moneyness_values = np.linspace(moneyness_min, moneyness_max, 30)

T_days = np.linspace(10, 730, 30)
T_years = T_days / 365.0

# Create meshgrid
moneyness_grid, T_grid = np.meshgrid(moneyness_values, T_years)
K_grid = S_fixed / moneyness_grid

# Flatten for model input
K_flat = K_grid.flatten()
T_flat = T_grid.flatten()
T_days_flat = (T_flat * 365.0).reshape(-1, 1)
vix_flat = np.full((len(T_flat), 1), vix_fixed)

to_scale = pd.DataFrame({"T": T_days_flat.flatten(), "vix": vix_flat.flatten()})
scaled_t_v = data_module.x_scaler.transform(to_scale)
T_scaled = scaled_t_v[:, 0]
vix_scaled = scaled_t_v[:, 1]

S_tensor = torch.full((len(K_flat), 1), S_fixed, dtype=torch.float32)
K_tensor = torch.tensor(K_flat, dtype=torch.float32).reshape(-1, 1)
T_tensor = torch.tensor(T_scaled, dtype=torch.float32).reshape(-1, 1)
vix_tensor = torch.tensor(vix_scaled, dtype=torch.float32).reshape(-1, 1)

hv_tensors = [
    torch.full((len(K_flat), 1), float(h_vol_values[col]), dtype=torch.float32)
    for col in hv_cols
]

# Concatenate [S, K, T_scaled, vix_scaled, hv_*]
x_input = torch.cat([S_tensor, K_tensor, T_tensor, vix_tensor, *hv_tensors], dim=1)

print("Predicting Prices...")
preds, greeks = model(x_input.to(device))

predicted_prices_scaled = preds.detach().cpu().numpy()
predicted_prices = (predicted_prices_scaled * K_tensor.numpy()).flatten()

delta_grid = greeks["delta"].detach().cpu().numpy().reshape(K_grid.shape)
gamma_grid = greeks["gamma"].detach().cpu().numpy().reshape(K_grid.shape)
theta_grid = greeks["theta"].detach().cpu().numpy().reshape(K_grid.shape)

price_grid = np.array(predicted_prices).reshape(K_grid.shape)


def plot_surface(z_grid, title, zaxis_title, colorbar_title):
    fig = go.Figure(data=[
        go.Surface(
            x=moneyness_grid,
            y=T_grid,
            z=z_grid,
            colorscale='Viridis',
            colorbar_title=colorbar_title,
            opacity=0.9,
            hovertemplate=(
                "Moneyness (S/K): %{x:.4f}<br>" +
                "Time (Years): %{y:.2f}<br>" +
                f"{zaxis_title}: %{{z:.6f}}<extra></extra>"
            )
        ),
    ])

    fig.update_layout(
        title={
            'text': title,
            'y': 0.9,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top'
        },
        scene=dict(
            xaxis_title='Moneyness (S/K)',
            yaxis_title='Time to Maturity (Years)',
            zaxis_title=zaxis_title,
            aspectratio=dict(x=1, y=1, z=0.6),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=0.5)
            )
        ),
        width=1000,
        height=700,
        margin=dict(l=50, r=50, b=0, t=50)
    )
    fig.show()


meta = f"date={target_date.date()}, S={S_fixed:.2f}, VIX={vix_fixed:.2f}"

print("Plotting Surface...")
plot_surface(
    price_grid,
    f"Price Surface ({meta})",
    'Price',
    'Predicted Price'
)

print("Plotting Delta Surface...")
plot_surface(
    delta_grid,
    f"Delta Surface ({meta})",
    'Delta',
    'Delta'
)

print("Plotting Gamma Surface...")
plot_surface(
    gamma_grid,
    f"Gamma Surface ({meta})",
    'Gamma',
    'Gamma'
)

print("Plotting Theta Surface...")
plot_surface(
    theta_grid,
    f"Theta Surface ({meta})",
    'Theta',
    'Theta'
)



Loading Data Module...
Train: 357036 | Validation: 44629 | Test: 44630
Loading Model from ./logs/my_experiment/version_82/checkpoints/epoch=49-step=69750.ckpt...
Loading raw data for selected date...
Using date=2025-01-02, S=5868.55, VIX=17.9300
Historical vol snapshot: {'hv_10': 0.129083, 'hv_14': 0.14111, 'hv_30': 0.146657, 'hv_60': 0.134958, 'hv_91': 0.126713}
Generating Grid...
Predicting Prices...
Plotting Surface...


Plotting Delta Surface...


Plotting Gamma Surface...


Plotting Theta Surface...
